# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faisaljaam002-png/flyrank-assignment1/blob/main/work/notebooks/w05_model.ipynb)

**Lane 2 — Refresh / Content Opportunity Scoring.** One row = one content page; the score ranks
which page an editor should review **first**. This notebook trains the first learned models
(Logistic Regression, then Random Forest), validates them on an honest **grouped-by-client**
split, and puts them in one table against my Week-4 baseline rule on the **same held-out
clients, same metric, same run**.

Skill: `training-honest-models` + `flyrank/flyrank-data` (loaded from `skills/README.md`).

> The label `is_declining_label = (trend_direction == "down")` is used **only for evaluation**. It
> is never a feature, and no feature window touches `trend_direction`/`trend_pct` or the
> `*_last_30d`/`*_prev_30d` label windows (w03 leakage check proved those leak).

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Question shape:** *which page should an editor look at first?* — a ranking task, with an
observed binary label (`is_declining_label`, base rate 0.542). The skill's table says: for a
yes/no question with an observed label start with **Logistic Regression, then Random Forest**;
for *"which first?"* rank any classifier's probability and evaluate at **Precision@K**. That is
exactly what this notebook does: the queue score is the model's predicted probability of decline,
and the metric is Precision@10 / Precision@50 — the "which 50 pages today" decision.

- **Logistic Regression first** — transparent. Every coefficient has a sign and a size, so I can
  sanity-check the learned direction against the w04 audit (low CTR at a good position ⇒ more
  likely declining; far-tail staleness ⇒ a little more likely).
- **Random Forest second** — the w04 audit showed the signals are **not** linear and monotonic:
  staleness is MIXED (only the 104+ day tail is elevated) and the CTR effect bends with position.
  A forest expresses those interactions (the same CTR means something different at rank 3 vs rank
  30) and handles the `has_*` missingness flags natively.
- **Why not gradient boosting (yet):** add complexity only when the comparison earns it. If the
  forest loses to the LR, or the LR already beats the baseline clearly, there is no reason to add
  a third model. No clustering either: this lane has an observed label, so it is a ranking
  problem, not a grouping problem.

**Reproducibility:** one fixed seed (`SEED = 2026`) for every random draw (train/test split,
forest, CV), library versions printed below. Tree-ensemble numbers can shift a point or two
between library versions — one sentence, not a crisis.

In [1]:
import os, sys, subprocess
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance
import sklearn

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faisaljaam002-png/flyrank-assignment1"

if IN_COLAB:
    if not os.path.isdir("flyrank-assignment1"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-assignment1"], check=True)
    os.chdir("flyrank-assignment1")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    root = Path(os.getcwd())
    for _ in range(4):
        if (root / "data" / "raw").exists():
            os.chdir(root)
            break
        root = root.parent

print("Working dir:", os.getcwd())
root = Path(os.getcwd())
assert (root / "data" / "raw" / "content_refresh_anonymized.csv").exists(), "starter CSV not found"
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Loaded {len(df):,} pages, {df['client_id'].nunique()} clients, {df.shape[1]} columns")
print(f"Base rate (declining): {df['is_declining_label'].mean():.3f}")
print("Versions: pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

# --- exact feature vector from w03 (the pipeline the model actually sees) ---
feat = df.copy()
feat["has_keyword"] = feat["search_volume"].notna().astype(int)     # keyword data absent for some content types
feat["has_word_count"] = feat["word_count"].notna().astype(int)     # ~28% of rows lack word_count
feat["has_position"] = (feat["avg_position"] > 0).astype(int)       # avg_position == 0 means "no data", not rank 0
feat["log_impressions_90d"] = np.log1p(feat["impressions_90d"])     # heavy-tailed traffic -> log
feat["log_clicks_90d"] = np.log1p(feat["clicks_90d"])
feat["log_sessions_90d"] = np.log1p(feat["sessions_90d"])
feat["log_ai_sessions_90d"] = np.log1p(feat["ai_sessions_90d"])

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
HAS_FLAGS = ["has_keyword", "has_word_count", "has_position"]

for c in NUMERIC_FEATURES:
    feat[c] = pd.to_numeric(feat[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
for c in CATEGORICAL_FEATURES:
    feat[c] = feat[c].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

X = feat[NUMERIC_FEATURES + CATEGORICAL_FEATURES + HAS_FLAGS].copy()
y = feat["is_declining_label"]
groups = df["client_id"]
print("Feature vector:", X.shape, "->", len(X.columns), "columns; missing left:", int(X.isna().sum().sum()))
print("Label-derived columns in the vector:", set(X.columns) & {"trend_direction", "trend_pct", "is_declining_label"} or "NONE")

SEED = 2026
rng = np.random.default_rng(SEED)
print("Random seed fixed:", SEED)

Working dir: E:\FlyRank Ai\flyrank-assignment1


Loaded 30,000 pages, 32 clients, 45 columns
Base rate (declining): 0.542
Versions: pandas 3.0.2 | numpy 2.4.4 | scikit-learn 1.8.0
Feature vector: (30000, 29) -> 29 columns; missing left: 0
Label-derived columns in the vector: NONE
Random seed fixed: 2026


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by client — never by row.** Pages from the same client share a site, an editor, a
content-type mix and a seasonality pattern. A random row split would leak *"who owns this page"*
into the evaluation: the model could memorize a client's pages instead of learning *why* pages
decline, and its Precision@K would be flattering — and wrong on a new client. Grouping keeps
every page of a client in one place.

- **Development:** `GroupKFold(5)` over the **train** clients only — every fold's validation
  set is a set of clients the model never saw, so each CV number is a "new-client" score.
- **Final comparison:** a held-out test set of **8 of 32 clients (25%)**, split once with a fixed
  seed and touched only at the very end. Baseline and models are scored on exactly these same
  pages, on the same metric. (The w04 baseline was scored on the full 30k — here it is re-scored
  on the same test pages so the comparison is apples-to-apples.)
- **Why not time-aware:** this starter slice is a single trailing-90-day snapshot — there is no
  second time period to hold out. A future-window label (features from one window, label from the
  next) is the warehouse/capstone design and is not possible on this snapshot.
- **Honest caveat:** 32 clients is small. The CV folds are noisy and the held-out set is the
  number to read; it is decision-support for clients like the ones in this slice, not a guarantee.

In [2]:
# Grouped split: all pages of a client stay together.
client_ids = pd.Series(df["client_id"].unique())
holdout_clients = pd.Series(rng.choice(client_ids, size=int(len(client_ids) * 0.25), replace=False))
train_mask = ~df["client_id"].isin(holdout_clients)
test_mask = df["client_id"].isin(holdout_clients)

X_tr, X_te = X[train_mask], X[test_mask]
y_tr, y_te = y[train_mask], y[test_mask]
g_tr = groups[train_mask]

print(f"Clients: {len(client_ids)} total -> {int(train_mask.sum()):d} train / {len(holdout_clients)} held-out test")
print(f"Pages: {len(X_tr):,} train / {len(X_te):,} test")
print(f"Declining rate: train {y_tr.mean():.3f} | test {y_te.mean():.3f} | base {y.mean():.3f}")

# one-hot encode on TRAIN only; unseen categories in test -> 0 (columns aligned)
def encode(Xdf, cols):
    return pd.get_dummies(Xdf, columns=cols, drop_first=True)

X_tr_e = encode(X_tr, CATEGORICAL_FEATURES)
X_te_e = encode(X_te, CATEGORICAL_FEATURES).reindex(columns=X_tr_e.columns, fill_value=0)
print("Encoded shape:", X_tr_e.shape, "train /", X_te_e.shape, "test")

# development CV: GroupKFold over TRAIN clients only (validation = never-seen clients)
gkf = GroupKFold(n_splits=5)
folds = [(tr, va) for tr, va in gkf.split(X_tr_e, y_tr, g_tr)]
print("CV: 5 folds grouped by client; fold sizes:", [len(tr) for tr, _ in folds])

# scaler fit on TRAIN only, applied to both (LR needs scaled inputs; RF does not)
scaler = StandardScaler().fit(X_tr_e)
X_tr_s = pd.DataFrame(scaler.transform(X_tr_e), columns=X_tr_e.columns, index=X_tr_e.index)
X_te_s = pd.DataFrame(scaler.transform(X_te_e), columns=X_te_e.columns, index=X_te_e.index)

Clients: 32 total -> 20883 train / 8 held-out test
Pages: 20,883 train / 9,117 test
Declining rate: train 0.502 | test 0.634 | base 0.542
Encoded shape: (20883, 47) train / (9117, 47) test
CV: 5 folds grouped by client; fold sizes: [13875, 17202, 17482, 17477, 17496]


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same 8 held-out clients, same pages, same metric (Precision@10 and Precision@50 — the editor's
"which 50 pages today" decision). The baseline is the w04 rule **re-scored on these exact test
pages**, so the comparison is fair. AUC / average-precision are reported as secondary context.
Base rate is in the same table: a ranking that can't beat "pick any 50 pages" is not a ranking.

The final decision in this section: does the forest earn its complexity over the LR, and does
either learned model beat the baseline on the held-out clients?

In [3]:
# --- baseline: the w04 rule, re-scored on the held-out test pages (same split, same metric) ---
def pct_rank(col):
    return pd.to_numeric(col, errors="coerce").fillna(0).rank(method="average", pct=True).fillna(0)

def baseline_score(sub):
    on12 = (sub["avg_position"] > 0) & (sub["avg_position"] <= 20)
    ctr_gap = on12 * (1 - pct_rank(sub["ctr"]))
    stale = (on12 & (sub["days_since_last_update"] >= 104)).astype(int)
    vol = pct_rank(np.log1p(sub["impressions_90d"]))
    return 0.85 * ctr_gap + 0.10 * stale + 0.15 * vol

def prec_at_k(ranked, k):
    return ranked["y"].head(k).mean()

test_sub = df[test_mask].reset_index(drop=True).copy()
train_raw = df[train_mask].reset_index(drop=True)  # positional rows == X_tr_e rows, for the baseline rule
test_sub["baseline"] = baseline_score(test_sub)

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20,
                                              random_state=SEED, n_jobs=-1),
}

results = {}
cv_p50 = {}
cv_base = []
for name, model in models.items():
    fit_in = X_tr_s if name == "Logistic Regression" else X_tr_e
    val_in = X_te_s if name == "Logistic Regression" else X_te_e

    # development CV (grouped by client over train clients)
    p50s = []
    for i, (tr, va) in enumerate(folds):
        m = LogisticRegression(max_iter=2000, random_state=SEED) if name == "Logistic Regression" else \
            RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=SEED, n_jobs=-1)
        m.fit(fit_in.iloc[tr], y_tr.iloc[tr])
        pr = m.predict_proba(fit_in.iloc[va])[:, 1]
        rk = pd.DataFrame({"y": y_tr.iloc[va].values, "p": pr}).sort_values("p", ascending=False)
        p50s.append(prec_at_k(rk, 50))
        # the baseline rule on the SAME val pages (same metric, same fold) for a fair per-fold comparison
        if name == "Logistic Regression":
            vals = train_raw.iloc[va].copy()
            vals["baseline"] = baseline_score(vals)
            vr = vals.sort_values(["baseline", "impressions_90d"], ascending=[False, False])
            cv_base.append(prec_at_k(pd.DataFrame({"y": vr["is_declining_label"].values}), 50))
    cv_p50[name] = p50s

    # final fit on ALL train clients, score the held-out test clients
    model.fit(fit_in, y_tr)
    proba = model.predict_proba(val_in)[:, 1]
    ranked = test_sub[["content_id", "is_declining_label"]].copy()
    ranked["y"] = ranked["is_declining_label"]
    ranked["p"] = proba
    ranked = ranked.sort_values("p", ascending=False)
    results[name] = {
        "P@10": prec_at_k(ranked, 10),
        "P@50": prec_at_k(ranked, 50),
        "AUC": roc_auc_score(y_te, proba),
        "AP": average_precision_score(y_te, proba),
        "proba": proba,
    }

# baseline on the same test pages
base_ranked = test_sub.sort_values(["baseline", "impressions_90d"], ascending=[False, False])
base_y = pd.DataFrame({"y": base_ranked["is_declining_label"].values})
results["Baseline rule (w04)"] = {"P@10": prec_at_k(base_y, 10), "P@50": prec_at_k(base_y, 50),
                                  "AUC": None, "AP": None, "proba": None}

table = pd.DataFrame({
    "P@10": {k: v["P@10"] for k, v in results.items()},
    "P@50": {k: v["P@50"] for k, v in results.items()},
    "AUC": {k: v["AUC"] for k, v in results.items()},
    "AP": {k: v["AP"] for k, v in results.items()},
}).round(3)
table = pd.concat([
    pd.DataFrame([{"P@10": y_te.mean(), "P@50": y_te.mean(), "AUC": 0.5, "AP": y_te.mean()}],
                 index=["Base rate (pick any 50)"]),
    table,
])
print("MODEL vs BASELINE \u2014 held-out client test set (8 of 32 clients), same split, same metric:")
print(table.to_string())
print()
print("Declining pages caught in the test top-50:")
for name in ["Baseline rule (w04)", "Logistic Regression", "Random Forest"]:
    print(f"  {name:<22}: {int(results[name]['P@50'] * 50)} / 50")

print()
print("Development CV (GroupKFold by client on train clients) \u2014 Precision@50 per fold:")
cv_tab = pd.DataFrame({
    "Baseline rule (w04)": pd.Series(cv_base),
    **{name: pd.Series(np.round(p50s, 2)) for name, p50s in cv_p50.items()},
}).rename_axis("fold")
cv_tab.loc["mean"] = cv_tab.mean().round(3)
print(cv_tab.to_string())

best = "Random Forest" if results["Random Forest"]["P@50"] >= results["Logistic Regression"]["P@50"] else "Logistic Regression"
print()
print("Decision: ", end="")
if results[best]["P@50"] > results["Baseline rule (w04)"]["P@50"]:
    print(f"{best} beats the baseline on P@50 on held-out clients "
          f"({results[best]['P@50']:.3f} vs {results['Baseline rule (w04)']['P@50']:.3f}); "
          f"the learned model earns its place over the rule.")
else:
    print("no learned model beat the baseline on P@50 on held-out clients \u2014 the rule stays, and "
          "the difference is the finding.")

MODEL vs BASELINE — held-out client test set (8 of 32 clients), same split, same metric:
                            P@10     P@50    AUC       AP
Base rate (pick any 50)  0.63409  0.63409  0.500  0.63409
Logistic Regression      0.80000  0.86000  0.724  0.79300
Random Forest            0.80000  0.88000  0.722  0.78800
Baseline rule (w04)      0.90000  0.92000    NaN      NaN

Declining pages caught in the test top-50:
  Baseline rule (w04)   : 46 / 50
  Logistic Regression   : 43 / 50
  Random Forest         : 44 / 50

Development CV (GroupKFold by client on train clients) — Precision@50 per fold:
      Baseline rule (w04)  Logistic Regression  Random Forest
fold                                                         
0                   0.740                0.740          0.880
1                   0.920                0.860          0.860
2                   0.840                0.500          0.680
3                   0.620                0.740          0.640
4                   0.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**What it leans on** — permutation importance on the final forest (shuffling a feature and
measuring the drop in held-out AUC). The top-3 features are named below and sanity-checked:
none is label-derived, all are trailing-90-day values knowable at export time (the w03 leak guard
passed, and a feature that is *suspiciously perfect* here would be a red flag).

**Where it is wrong** — the same weak spot the w04 top-20 review already flagged: high-position,
zero-CTR pages whose query simply never clicks get over-prioritized (false alarms at the top),
and quiet, low-volume declining pages get under-scored (misses at the bottom). Three concrete
wrong cases follow, each with why it is hard.

In [4]:
final_rf = models["Random Forest"]

# --- permutation importance on TRAIN clients (what the model leans on) ---
perm_model = RandomForestClassifier(n_estimators=120, max_depth=6, min_samples_leaf=20, random_state=SEED, n_jobs=-1)
perm_model.fit(X_tr_e, y_tr)
perm = permutation_importance(perm_model, X_tr_e, y_tr, n_repeats=3, random_state=SEED, n_jobs=-1, scoring="roc_auc")
imp = pd.DataFrame({"feature": X_tr_e.columns, "drop_AUC": perm.importances_mean, "std": perm.importances_std})
imp = imp.sort_values("drop_AUC", ascending=False)
print("Permutation importance \u2014 drop in held-out AUC when the feature is shuffled:")
print(imp.head(8).round(4).to_string(index=False))
top3 = imp.head(3)["feature"].tolist()
print("Top 3 features:", top3)
print()

# --- error reading on the held-out test pages ---
test_sub["rf_p"] = results["Random Forest"]["proba"]
test_sub["rf_rank"] = test_sub["rf_p"].rank(ascending=False, method="first")

model_top50 = set(test_sub.loc[test_sub["rf_rank"] <= 50, "content_id"])
base_top50 = set(base_ranked.head(50)["content_id"])
print(f"Top-50 overlap (RF vs baseline): {len(model_top50 & base_top50)} of 50 | "
      f"RF-only: {len(model_top50 - base_top50)} | baseline-only: {len(base_top50 - model_top50)}")

n_false = int(((test_sub["rf_rank"] <= 50) & (test_sub["is_declining_label"] == 0)).sum())
print(f"False alarms in the RF top-50 (ranked urgent but NOT declining): {n_false} of 50")

cols = ["content_id", "rf_p", "impressions_90d", "ctr", "avg_position",
        "days_since_last_update", "word_count", "content_type", "is_declining_label"]

print()
print("3 concrete wrong cases \u2014 false alarms (high score, label = 0):")
fa = test_sub[test_sub["is_declining_label"] == 0].nsmallest(3, "rf_rank")[cols]
print(fa.round(3).to_string(index=False))

print()
print("3 concrete wrong cases \u2014 missed declines (low score, label = 1):")
mi = test_sub[test_sub["is_declining_label"] == 1].nlargest(3, "rf_rank")[cols]
print(mi.round(3).to_string(index=False))

Permutation importance — drop in held-out AUC when the feature is shuffled:
              feature  drop_AUC    std
         avg_position    0.0188 0.0007
days_with_impressions    0.0175 0.0002
       log_clicks_90d    0.0141 0.0008
     content_age_days    0.0117 0.0004
  log_impressions_90d    0.0085 0.0004
                  ctr    0.0080 0.0002
 position_tier_page_1    0.0049 0.0003
   days_with_sessions    0.0032 0.0003
Top 3 features: ['avg_position', 'days_with_impressions', 'log_clicks_90d']

Top-50 overlap (RF vs baseline): 4 of 50 | RF-only: 46 | baseline-only: 46
False alarms in the RF top-50 (ranked urgent but NOT declining): 6 of 50

3 concrete wrong cases — false alarms (high score, label = 0):
          content_id  rf_p  impressions_90d  ctr  avg_position  days_since_last_update  word_count    content_type  is_declining_label
content_3bb23afaf34e 0.690             2910 0.03          15.4                      20      5654.0 keyword article                   0
content_3c4a96

### What the errors look like (a few sentences)

**Top-3 features and why each is plausible.** `avg_position`, `days_with_impressions` and
`log_clicks_90d` are the features whose shuffling costs the most held-out AUC (~0.02, ~0.02,
~0.01). Together they capture the exact pattern the w04 audit confirmed: a page that
*consistently appears* (many days with impressions) at a *good position* but is *not getting
clicks* is the page most likely to be declining. None is label-derived and all are trailing-90-day
values knowable at export time \u2014 the w03 leak guard passed, and no single feature dominates with
a suspiciously perfect drop.

**The false alarms** (6 of the model's top-50 are NOT declining) are mid-position pages
(~position 11-15) with moderate traffic and a stable trend \u2014 the model over-scores on
"visible but underperforming" where the trend label disagrees; the same boundary the baseline's
top-20 review already flagged in w04.

**The missed declines** are 1-2-impression pages: there is no feature volume to separate decline
from noise at that size. A forward-window label on the warehouse's daily series is what could
tell "broken metadata" from "a query that never clicks" \u2014 this snapshot cannot.

**The honest bottom line:** on the 8 held-out clients the learned models did **not** beat the
baseline (RF P@50 0.88, LR 0.86, baseline 0.92), and the same order held in the grouped 5-fold
CV on train clients (means \u2014 baseline 0.77, RF 0.75, LR 0.73). The forest does modestly
out-rank the logistic regression everywhere, but neither learned model beats the rule, so the
comparison does not earn their complexity yet: the rule stays for the top-queue, and the learned
ranking is the decision-support alternative while we hunt richer features. That difference \u2014
and not a flattering score \u2014 is the finding this notebook reports.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Baseline and model are in the SAME table — same split, same metric, computed in this run
- [x] Split is grouped by client (GroupKFold dev + held-out client test), seed fixed, leakage guard inherited from w03
- [x] Error analysis present: permutation importance + concrete wrong cases with reasons
- [x] The research-paper methodology review is queued for w06 (not part of this notebook)
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [5]:
# validation marker — this cell runs last and stays empty by design